In [ ]:
import pandas as pd
import numpy as np
import gensim.downloader as api

from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense
from gensim.utils import simple_preprocess
from gensim.models import Word2Vec


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
MAX_WORDS = 100000
MAX_LEN = 40
EMBEDDING_DIM = 50
NUM_CLASSES = 6


In [4]:
# load data
df = pd.read_csv("twitter_emotion.csv")

texts = df["text"].astype(str)
labels = df["label"]

In [ ]:
# Keras Tokenization
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN,
                                 padding="post", truncating="post")

word_index = tokenizer.word_index
vocab_size = min(MAX_WORDS, len(word_index) + 1)

In [7]:
# one-hot encoding
y = to_categorical(labels, num_classes=NUM_CLASSES)

#split data
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, y,test_size=0.20, random_state=42)

# Glove Model

[==================================================] 100.0% 199.5/199.5MB downloaded


In [9]:
embedding_matrix_glove = np.zeros((vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i >= vocab_size:
        continue

    if word in glove_model:
        embedding_matrix_glove[i] = glove_model[word]

In [ ]:
# build cnn model with glove embedding matrix
model_glove = Sequential()

model_glove.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix_glove],
        trainable=False
    )
)

model_glove.add(Conv1D(128, 5, activation="relu"))
model_glove.add(Conv1D(64, 5, activation="relu"))
model_glove.add(GlobalMaxPooling1D())
model_glove.add(Dense(NUM_CLASSES, activation="softmax"))

model_glove.compile(loss="categorical_crossentropy", optimizer="adam",
                    metrics=["accuracy"])

In [11]:
model_glove.fit(X_train, y_train, epochs=5,
    batch_size=128, validation_data=(X_test, y_test))

Epoch 1/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 30s 11ms/step - accuracy: 0.8146 - loss: 0.5140 - val_accuracy: 0.8905 - val_loss: 0.2908
Epoch 2/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 30s 12ms/step - accuracy: 0.9008 - loss: 0.2495 - val_accuracy: 0.9023 - val_loss: 0.2301
Epoch 3/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 26s 10ms/step - accuracy: 0.9108 - loss: 0.2091 - val_accuracy: 0.8907 - val_loss: 0.2702
Epoch 4/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 11ms/step - accuracy: 0.9155 - loss: 0.1905 - val_accuracy: 0.9052 - val_loss: 0.2139
Epoch 5/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 28s 11ms/step - accuracy: 0.9198 - loss: 0.1775 - val_accuracy: 0.9067 - val_loss: 0.2080


In [12]:
loss, glove_accuracy = model_glove.evaluate(X_test, y_test)

print("GloVe Accuracy:", glove_accuracy)

2606/2606 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9067 - loss: 0.2080
GloVe Accuracy: 0.9066600799560547


# WORD2VEC Model

In [6]:
# Gensim Tokenization
tokenized_tweets = []
for tweet in texts:
    tokens = simple_preprocess(tweet)
    tokenized_tweets.append(tokens)

In [13]:
w2v_model = Word2Vec(sentences=tokenized_tweets, vector_size=EMBEDDING_DIM,
    window=5, min_count=2,workers=4)

In [14]:
embedding_matrix_w2v = np.zeros((vocab_size, EMBEDDING_DIM))

for word, i in word_index.items():
    if i >= vocab_size:
        continue

    if word in w2v_model.wv:
        embedding_matrix_w2v[i] = w2v_model.wv[word]

In [ ]:
# build cnn model with w2v embedding matrix
model_w2v = Sequential()

model_w2v.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix_w2v],
        trainable=False
    )
)

model_w2v.add(Conv1D(128, 5, activation="relu"))
model_w2v.add(Conv1D(64, 5, activation="relu"))
model_w2v.add(GlobalMaxPooling1D())
model_w2v.add(Dense(NUM_CLASSES, activation="softmax"))

model_w2v.compile(loss="categorical_crossentropy", optimizer="adam",
    metrics=["accuracy"]
)

In [16]:
model_w2v.fit(X_train, y_train, epochs=5,
    batch_size=128, validation_data=(X_test, y_test))

Epoch 1/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - accuracy: 0.8519 - loss: 0.3944 - val_accuracy: 0.9029 - val_loss: 0.2543
Epoch 2/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - accuracy: 0.9094 - loss: 0.2186 - val_accuracy: 0.9105 - val_loss: 0.2065
Epoch 3/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - accuracy: 0.9175 - loss: 0.1898 - val_accuracy: 0.9125 - val_loss: 0.1958
Epoch 4/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - accuracy: 0.9210 - loss: 0.1742 - val_accuracy: 0.9136 - val_loss: 0.1970
Epoch 5/5
2606/2606 ━━━━━━━━━━━━━━━━━━━━ 27s 10ms/step - accuracy: 0.9233 - loss: 0.1634 - val_accuracy: 0.9144 - val_loss: 0.1851


In [17]:
loss, w2v_accuracy = model_w2v.evaluate(X_test, y_test)

print("Word2Vec Accuracy:", w2v_accuracy)

2606/2606 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9144 - loss: 0.1851
Word2Vec Accuracy: 0.9144334197044373


In [22]:
print("==== FINAL RESULTS ====")
print("GloVe Accuracy:", glove_accuracy)
print("Word2Vec Accuracy:", w2v_accuracy)

==== FINAL RESULTS ====
GloVe Accuracy: 0.9066600799560547
Word2Vec Accuracy: 0.9144334197044373


# User Input

In [29]:
# Emotion mapping
label_map = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

def predict_emotion(text, model):
    sequence = tokenizer.texts_to_sequences([text])

    padded = pad_sequences(sequence, maxlen=MAX_LEN, padding="post",
                           truncating="post")
    prediction = model.predict(padded)
    predicted_class = np.argmax(prediction)

    return label_map[predicted_class]

In [40]:
user_text = "I am so happy today!"

if glove_accuracy >= w2v_accuracy:
    emotion = predict_emotion(user_text, model_glove)
    print("Using GloVe model")
    print("Predicted Emotion:", emotion)
else:
    emotion = predict_emotion(user_text, model_w2v)
    print("Using Word2Vec model")
    print("Predicted Emotion:", emotion)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
Using Word2Vec model
Predicted Emotion: joy
